# E2 — YOLOv10n với data augmentation

E2 giữ nguyên model, split, seed và thiết lập huấn luyện của E1. Khác biệt duy nhất là augmentation chỉ trên train: brightness/contrast, Gaussian noise, Gaussian blur, cùng scale/translation mạnh hơn. Validation và test không bị biến đổi.

Chuẩn bị từ repository root:

```bash
uv sync --extra dev --extra train
uv pip install git+https://github.com/THU-MIG/yolov10.git
uv run --with jupyterlab jupyter lab
```

Cần đặt pretrained weight tại `weights/yolov10n.pt` và chuẩn bị dataset theo `configs/data.yaml`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from pprint import pprint

candidate = Path.cwd().resolve()
for directory in (candidate, *candidate.parents):
    if (directory / 'pyproject.toml').is_file():
        PROJECT_ROOT = directory
        break
else:
    raise RuntimeError('Không tìm thấy project root (pyproject.toml).')

os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

E2_CONFIG = PROJECT_ROOT / 'configs' / 'E2_augmentation.yaml'
DATA_CONFIG = PROJECT_ROOT / 'configs' / 'data.yaml'
WEIGHTS = PROJECT_ROOT / 'weights' / 'yolov10n.pt'
print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.executable}')

## 1. Kiểm tra E2 protocol

Các giá trị trong `augmentation` là custom photometric transforms. `scale`, `translate`, và `close_mosaic` là tham số native của YOLO được E2 override từ E1.

In [ ]:
from helmet_yolov10.augmentation.enhanced import parse_e2_augmentation
from helmet_yolov10.utils.config import load_config

config = load_config(E2_CONFIG)
assert config['experiment']['id'] == 'E2'
settings = parse_e2_augmentation(config['augmentation'])

pprint({
    'experiment': config['experiment'],
    'custom_augmentation': settings,
    'native_yolo_changes': {key: config['training'][key] for key in ('scale', 'translate', 'close_mosaic')},
})

## 2. Validate dataset

E2 dùng chính xác dataset split của E1. Nếu cell này lỗi, sửa dataset/config trước khi train.

In [ ]:
from helmet_yolov10.data.validation import validate_dataset

dataset_report = validate_dataset(DATA_CONFIG, PROJECT_ROOT)
pprint(dataset_report)

## 3. Kiểm tra backend, Albumentations và pretrained weight

In [ ]:
import albumentations

from helmet_yolov10.training.train import _load_backend
from helmet_yolov10.utils.environment import sha256_file

if not WEIGHTS.is_file():
    raise FileNotFoundError(f'Không tìm thấy pretrained weight: {WEIGHTS}')

print(f'Albumentations: {albumentations.__version__}')
print(f'YOLO backend: {_load_backend()}')
print(f'Weight SHA-256: {sha256_file(WEIGHTS)}')

## 4. Dry run

Dry run xác nhận config E2 và dataset nhưng không load model, không dùng GPU, không tạo run directory.

In [ ]:
from helmet_yolov10.training.train import train_experiment

train_experiment(E2_CONFIG, validate_only=True)
print('E2 dry run thành công.')

## 5. Train E2

Đổi `RUN_TRAIN` thành `True` sau khi tất cả các cell kiểm tra thành công. Tên run phải là duy nhất; trainer không ghi đè kết quả cũ.

In [ ]:
RUN_TRAIN = False
DEVICE = 0
RUN_NAME = 'augmentation_seed42'

if RUN_TRAIN:
    run_dir = train_experiment(
        E2_CONFIG,
        run_name=RUN_NAME,
        device=DEVICE,
    )
    print(f'Train E2 hoàn tất: {run_dir}')
else:
    print('Chưa train. Đổi RUN_TRAIN thành True rồi chạy lại cell này.')

## 6. Đánh giá E2 trên test set không augmentation

Đánh giá dùng `best.pt` và giữ test set nguyên trạng để so sánh công bằng với E1.

In [ ]:
from helmet_yolov10.evaluation.evaluate import evaluate_checkpoint

RUN_EVALUATION = False
CHECKPOINT = PROJECT_ROOT / 'experiments' / 'E2' / RUN_NAME / 'weights' / 'best.pt'
EVALUATION_RUN_NAME = f'{RUN_NAME}_test'

if RUN_EVALUATION:
    if not CHECKPOINT.is_file():
        raise FileNotFoundError(f'Không tìm thấy checkpoint: {CHECKPOINT}')
    evaluation_dir = evaluate_checkpoint(
        CHECKPOINT, E2_CONFIG, run_name=EVALUATION_RUN_NAME, device=DEVICE
    )
    print(f'Đánh giá E2 hoàn tất: {evaluation_dir}')
else:
    print('Chưa đánh giá. Đổi RUN_EVALUATION thành True rồi chạy lại cell này.')